# BoneGAN: Bone-Conducted Speech Enhancement Demo

This notebook serves as an interactive walkthrough of the BoneGAN pipeline using the modular codebase (`data.py`, `model.py`, `losses.py`, `train.py`, `evaluate.py`).

> **Attribution:** Generator and discriminator adapted from the TensorFlow pix2pix tutorial; adapted to paired bone/air-conducted speech spectrograms.

In [ ]:
# 1. Environment Setup & Reproducibility
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from utils import load_config, set_seed
from model import Generator, Discriminator
from losses import generator_loss, discriminator_loss
from data import extract_slices, prepare_3d_images, slices_to_magnitude, reconstruct_waveform

config = load_config("config.yaml")
set_seed(config.get("seed", 42))
print("TensorFlow Version:", tf.__version__)
print("Loaded configuration with seed:", config.get("seed", 42))

## 2. Inspect Model Architectures
Instantiate the U-Net Generator and PatchGAN Discriminator from `model.py`.

In [ ]:
generator = Generator(input_shape=(256, 256, 3), output_channels=3)
discriminator = Discriminator(input_shape=(256, 256, 3))

print("Generator output shape:", generator.output_shape)
print("Discriminator output shape:", discriminator.output_shape)

## 3. Data Processing on Synthetic Paired Audio
Generate a synthetic clean speech harmonic signal and its low-pass filtered "bone" counterpart to demonstrate data extraction without requiring the private dataset.

In [ ]:
from scipy import signal

sr = 16000
duration = 1.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Multi-harmonic clean signal
clean_audio = (
    0.5 * np.sin(2 * np.pi * 300 * t)
    + 0.3 * np.sin(2 * np.pi * 800 * t)
    + 0.15 * np.sin(2 * np.pi * 2200 * t)
)

# Simulated bone attenuation (low-pass filter <1kHz)
sos = signal.butter(4, 1000.0, btype="lowpass", fs=sr, output="sos")
bone_audio = signal.sosfilt(sos, clean_audio) + 0.01 * np.random.randn(len(clean_audio))

# Extract STFT slices and build 256x256x3 spectrogram images
slices_c, slices_n, phase_bone, _ = extract_slices(
    clean_audio=clean_audio,
    noise_audio=bone_audio,
    sr=sr,
    nfft=256,
    noverlap=192,
)
inputs, targets, stats_n, stats_c = prepare_3d_images(
    slices_c=slices_c,
    slices_n=slices_n,
    img_height=256,
    img_width=256,
)
print(f"Prepared {len(inputs)} spectrogram image pairs of shape {inputs[0].shape}")

## 4. Run Single Gradient Step
Demonstrates the loss calculation from `losses.py` and model update with `tf.GradientTape`.

In [ ]:
gen_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
disc_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
    gen_out = generator(inputs[:1], training=True)
    disc_real = discriminator([inputs[:1], targets[:1]], training=True)
    disc_gen = discriminator([inputs[:1], gen_out], training=True)

    total_gen_loss, gan_loss, l1_loss = generator_loss(disc_gen, gen_out, targets[:1], lambda_l1=100.0)
    disc_loss = discriminator_loss(disc_real, disc_gen)

gen_grads = gen_tape.gradient(total_gen_loss, generator.trainable_variables)
disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
gen_opt.apply_gradients(zip(gen_grads, generator.trainable_variables))
disc_opt.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

print(f"Gen Total Loss: {float(total_gen_loss):.4f} (GAN: {float(gan_loss):.4f}, L1: {float(l1_loss):.4f})")
print(f"Disc Loss: {float(disc_loss):.4f}")

## 5. Waveform Reconstruction & Metric Evaluation
Invert spectrogram prediction back to time-domain audio and compute PESQ, STOI, and LSD.

In [ ]:
from pesq import pesq
from pystoi import stoi
from evaluate import compute_lsd

pred_3d = generator(inputs, training=False).numpy()
pred_mag = slices_to_magnitude(pred_3d, stats=stats_c, orig_f_bins=129, reps_horizontal=8)
pred_waveform = reconstruct_waveform(pred_mag, phase=phase_bone, sr=sr)

min_len = min(len(clean_audio), len(pred_waveform))
clean_cut = clean_audio[:min_len]
pred_cut = pred_waveform[:min_len]

pesq_val = pesq(sr, clean_cut, pred_cut, "wb")
stoi_val = stoi(clean_cut, pred_cut, sr, extended=False)
lsd_val = compute_lsd(clean_cut, pred_cut, sr=sr)

print(f"Demo Evaluation Metrics: PESQ={pesq_val:.4f}, STOI={stoi_val:.4f}, LSD={lsd_val:.4f} dB")

## 6. Command Line Workflow
To train on your dataset:
```bash
python train.py --config config.yaml
```

To evaluate the test split:
```bash
python evaluate.py --checkpoint training_checkpoints --config config.yaml
```

To run the automated smoke test suite:
```bash
pytest tests/test_pipeline.py
```